# Ecological Metadata Feature Extraction

The goal of this notebook is to extract ecological/context metadata from the BirdCLEF 2026 soundscape identifiers.

For example, from this row_id:

`BC2026_Train_0001_S08_20250606_030007_5`

we can extract:

- site = S08
- date = 2025-06-06
- time = 03:00:07
- start_second = 0
- end_second = 5

This notebook does not use audio features. It only extracts metadata from filenames and row IDs.

Later, these features can be used to test how much bird presence can be predicted from ecological/context information alone.

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
def find_competition_dir():
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/competitions/birdclef-2026"),
    ]

    for p in candidates:
        if p.exists():
            return p

    for p in Path("/kaggle/input").iterdir():
        if p.is_dir() and "birdclef" in p.name.lower():
            return p

    raise FileNotFoundError("Could not find BirdCLEF 2026 input folder.")

BASE = find_competition_dir()
print("Using:", BASE)

for f in sorted(BASE.iterdir()):
    print(f.name)

Using: /kaggle/input/competitions/birdclef-2026
recording_location.txt
sample_submission.csv
taxonomy.csv
test_soundscapes
train.csv
train_audio
train_soundscapes
train_soundscapes_labels.csv


In [3]:
csv_files = list(BASE.glob("*.csv"))

print("CSV files found:")
for f in csv_files:
    print("-", f.name)

sample_sub = pd.read_csv(BASE / "sample_submission.csv") if (BASE / "sample_submission.csv").exists() else None
taxonomy = pd.read_csv(BASE / "taxonomy.csv") if (BASE / "taxonomy.csv").exists() else None

if (BASE / "train_soundscapes_labels.csv").exists():
    soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
else:
    soundscape_labels = None

if sample_sub is not None:
    print("sample_submission shape:", sample_sub.shape)
    display(sample_sub.head())

if taxonomy is not None:
    print("taxonomy shape:", taxonomy.shape)
    display(taxonomy.head())

if soundscape_labels is not None:
    print("train_soundscapes_labels shape:", soundscape_labels.shape)
    display(soundscape_labels.head())

CSV files found:
- sample_submission.csv
- taxonomy.csv
- train_soundscapes_labels.csv
- train.csv
sample_submission shape: (3, 235)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


taxonomy shape: (234, 5)


,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia


train_soundscapes_labels shape: (1478, 4)


,filename,start,end,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380


In [4]:
ROW_ID_RE = re.compile(
    r"BC2026_(Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})_(\d+)"
)

def parse_row_id(row_id):
    m = ROW_ID_RE.match(row_id)

    if not m:
        return {
            "split": None,
            "file_id": None,
            "site": None,
            "date_str": None,
            "time_str": None,
            "datetime": pd.NaT,
            "year": np.nan,
            "month": np.nan,
            "day": np.nan,
            "hour": np.nan,
            "minute": np.nan,
            "second": np.nan,
            "day_of_year": np.nan,
            "end_second": np.nan,
            "start_second": np.nan,
            "window_index": np.nan,
        }

    split, file_id, site, ymd, hms, end_sec = m.groups()

    dt = pd.to_datetime(ymd + hms, format="%Y%m%d%H%M%S")
    end_sec = int(end_sec)
    start_sec = max(0, end_sec - 5)

    return {
        "split": split,
        "file_id": file_id,
        "site": site,
        "date_str": ymd,
        "time_str": hms,
        "datetime": dt,
        "year": dt.year,
        "month": dt.month,
        "day": dt.day,
        "hour": dt.hour,
        "minute": dt.minute,
        "second": dt.second,
        "day_of_year": dt.dayofyear,
        "end_second": end_sec,
        "start_second": start_sec,
        "window_index": start_sec // 5,
    }

In [5]:
example_row_id = "BC2026_Train_0001_S08_20250606_030007_5"

example_features = parse_row_id(example_row_id)

for k, v in example_features.items():
    print(f"{k}: {v}")

split: Train
file_id: 0001
site: S08
date_str: 20250606
time_str: 030007
datetime: 2025-06-06 03:00:07
year: 2025
month: 6
day: 6
hour: 3
minute: 0
second: 7
day_of_year: 157
end_second: 5
start_second: 0
window_index: 0


In [6]:
FILENAME_RE = re.compile(
    r"BC2026_(Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg"
)

def parse_filename(filename):
    filename = str(filename).split("/")[-1]
    m = FILENAME_RE.match(filename)

    if not m:
        return {
            "split": None,
            "file_id": None,
            "site": None,
            "date_str": None,
            "time_str": None,
            "datetime": pd.NaT,
            "year": np.nan,
            "month": np.nan,
            "day": np.nan,
            "hour": np.nan,
            "minute": np.nan,
            "second": np.nan,
            "day_of_year": np.nan,
        }

    split, file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd + hms, format="%Y%m%d%H%M%S")

    return {
        "split": split,
        "file_id": file_id,
        "site": site,
        "date_str": ymd,
        "time_str": hms,
        "datetime": dt,
        "year": dt.year,
        "month": dt.month,
        "day": dt.day,
        "hour": dt.hour,
        "minute": dt.minute,
        "second": dt.second,
        "day_of_year": dt.dayofyear,
    }

In [7]:
def to_seconds(x):
    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(x)

    try:
        return int(pd.to_timedelta(x).total_seconds())
    except Exception:
        return int(x)

def parse_label_list(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

def union_labels(series):
    labels = set()
    for x in series:
        labels.update(parse_label_list(x))
    return sorted(labels)

if soundscape_labels is None:
    raise ValueError("train_soundscapes_labels.csv was not found.")

# Aggregate labels for each 5-second window
train_meta = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

train_meta["start_second"] = train_meta["start"].apply(to_seconds)
train_meta["end_second"] = train_meta["end"].apply(to_seconds)

train_meta["row_id"] = (
    train_meta["filename"].str.replace(".ogg", "", regex=False)
    + "_"
    + train_meta["end_second"].astype(str)
)

parsed = train_meta["row_id"].apply(parse_row_id).apply(pd.Series)

# Drop duplicate start/end from parser if already present
parsed = parsed.drop(columns=["start_second", "end_second"], errors="ignore")

train_meta = pd.concat([train_meta, parsed], axis=1)

# Use start/end from actual label file
train_meta["window_index"] = train_meta["start_second"] // 5

display(train_meta.head())
print("Rows:", len(train_meta))
print("Unique files:", train_meta["filename"].nunique())
print("Unique sites:", train_meta["site"].nunique())

,filename,start,end,label_list,start_second,end_second,row_id,split,file_id,site,...,time_str,datetime,year,month,day,hour,minute,second,day_of_year,window_index
0,BC2026_Train_0001_S08_20250606_030007.ogg,00:00:00,00:00:05,"[47158son13, 47158son17, 47158son22, 47158son2...",0,5,BC2026_Train_0001_S08_20250606_030007_5,Train,0001,S08,...,030007,2025-06-06 03:00:07,2025,6,6,3,0,7,157,0
1,BC2026_Train_0001_S08_20250606_030007.ogg,00:00:05,00:00:10,"[47158son13, 47158son17, 47158son21, 47158son2...",5,10,BC2026_Train_0001_S08_20250606_030007_10,Train,0001,S08,...,030007,2025-06-06 03:00:07,2025,6,6,3,0,7,157,1
2,BC2026_Train_0001_S08_20250606_030007.ogg,00:00:10,00:00:15,"[47158son13, 47158son17, 47158son21, 47158son2...",10,15,BC2026_Train_0001_S08_20250606_030007_15,Train,0001,S08,...,030007,2025-06-06 03:00:07,2025,6,6,3,0,7,157,2
3,BC2026_Train_0001_S08_20250606_030007.ogg,00:00:15,00:00:20,"[47158son13, 47158son17, 47158son21, 47158son2...",15,20,BC2026_Train_0001_S08_20250606_030007_20,Train,0001,S08,...,030007,2025-06-06 03:00:07,2025,6,6,3,0,7,157,3
4,BC2026_Train_0001_S08_20250606_030007.ogg,00:00:20,00:00:25,"[47158son10, 47158son13, 47158son17, 47158son2...",20,25,BC2026_Train_0001_S08_20250606_030007_25,Train,0001,S08,...,030007,2025-06-06 03:00:07,2025,6,6,3,0,7,157,4


Rows: 739
Unique files: 66
Unique sites: 9


In [8]:
train_meta["hour_sin"] = np.sin(2 * np.pi * train_meta["hour"] / 24)
train_meta["hour_cos"] = np.cos(2 * np.pi * train_meta["hour"] / 24)

train_meta["month_sin"] = np.sin(2 * np.pi * train_meta["month"] / 12)
train_meta["month_cos"] = np.cos(2 * np.pi * train_meta["month"] / 12)

train_meta["dayofyear_sin"] = np.sin(2 * np.pi * train_meta["day_of_year"] / 365)
train_meta["dayofyear_cos"] = np.cos(2 * np.pi * train_meta["day_of_year"] / 365)

feature_cols = [
    "row_id",
    "filename",
    "site",
    "datetime",
    "year",
    "month",
    "day",
    "hour",
    "minute",
    "second",
    "day_of_year",
    "start_second",
    "end_second",
    "window_index",
    "label_list",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",
]

display(train_meta[feature_cols].head(20))

,row_id,filename,site,datetime,year,month,day,hour,minute,second,...,start_second,end_second,window_index,label_list,hour_sin,hour_cos,month_sin,month_cos,dayofyear_sin,dayofyear_cos
0,BC2026_Train_0001_S08_20250606_030007_5,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,0,5,0,"[47158son13, 47158son17, 47158son22, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
1,BC2026_Train_0001_S08_20250606_030007_10,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,5,10,1,"[47158son13, 47158son17, 47158son21, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
2,BC2026_Train_0001_S08_20250606_030007_15,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,10,15,2,"[47158son13, 47158son17, 47158son21, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
3,BC2026_Train_0001_S08_20250606_030007_20,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,15,20,3,"[47158son13, 47158son17, 47158son21, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
4,BC2026_Train_0001_S08_20250606_030007_25,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,20,25,4,"[47158son10, 47158son13, 47158son17, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
5,BC2026_Train_0001_S08_20250606_030007_30,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,25,30,5,"[47158son10, 47158son13, 47158son17, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
6,BC2026_Train_0001_S08_20250606_030007_35,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,30,35,6,"[47158son10, 47158son13, 47158son17, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
7,BC2026_Train_0001_S08_20250606_030007_40,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,35,40,7,"[47158son10, 47158son13, 47158son17, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
8,BC2026_Train_0001_S08_20250606_030007_45,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,40,45,8,"[47158son10, 47158son13, 47158son17, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193
9,BC2026_Train_0001_S08_20250606_030007_50,BC2026_Train_0001_S08_20250606_030007.ogg,S08,2025-06-06 03:00:07,2025,6,6,3,0,7,...,45,50,9,"[47158son10, 47158son13, 47158son17, 47158son2...",0.707107,0.707107,1.224647e-16,-1.0,0.425000,-0.905193


In [9]:
example_row_id = "BC2026_Train_0001_S08_20250606_030007_5"

example = train_meta[train_meta["row_id"] == example_row_id]

if len(example) == 0:
    print("Example row_id not found in train_meta.")
else:
    display(example[feature_cols].T)

,0
row_id,BC2026_Train_0001_S08_20250606_030007_5
filename,BC2026_Train_0001_S08_20250606_030007.ogg
site,S08
datetime,2025-06-06 03:00:07
year,2025
month,6
day,6
hour,3
minute,0
second,7


In [10]:
print("Rows:", len(train_meta))
print("Unique soundscape files:", train_meta["filename"].nunique())
print("Unique sites:", train_meta["site"].nunique())
print("Date range:", train_meta["datetime"].min(), "to", train_meta["datetime"].max())

print("\nRows by site:")
display(train_meta["site"].value_counts().sort_index())

print("\nRows by hour:")
display(train_meta["hour"].value_counts().sort_index())

print("\nRows by month:")
display(train_meta["month"].value_counts().sort_index())

Rows: 739
Unique soundscape files: 66
Unique sites: 9
Date range: 2021-10-16 01:15:00 to 2025-08-31 00:00:00

Rows by site:


site
S03     24
S08     60
S09     19
S13     24
S15     48
S18     15
S19     36
S22    477
S23     36
Name: count, dtype: int64


Rows by hour:


hour
0      52
1      51
2      60
3      48
4      36
6      48
7      36
18     12
19     36
20     84
21     96
22     72
23    108
Name: count, dtype: int64


Rows by month:


month
1     108
2     144
4      12
6     108
8      19
10     27
11    165
12    156
Name: count, dtype: int64

In [11]:
exploded = train_meta.explode("label_list").rename(columns={"label_list": "primary_label"})

species_site_counts = (
    exploded
    .dropna(subset=["primary_label"])
    .groupby(["primary_label", "site"])
    .size()
    .reset_index(name="count")
    .sort_values(["primary_label", "count"], ascending=[True, False])
)

display(species_site_counts.head(30))

,primary_label,site,count
0,116570,S08,10
1,116570,S23,3
4,1491113,S22,53
2,1491113,S03,14
3,1491113,S13,12
6,22961,S22,24
5,22961,S13,12
10,22967,S22,120
7,22967,S03,12
9,22967,S19,12


In [12]:
output_cols = [
    "row_id",
    "filename",
    "site",
    "datetime",
    "year",
    "month",
    "day",
    "hour",
    "minute",
    "second",
    "day_of_year",
    "start_second",
    "end_second",
    "window_index",
    "label_list",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",
]

train_meta[output_cols].to_csv(
    "/kaggle/working/train_ecological_metadata_features.csv",
    index=False
)

species_site_counts.to_csv(
    "/kaggle/working/species_site_counts.csv",
    index=False
)

print("Saved:")
print("/kaggle/working/train_ecological_metadata_features.csv")
print("/kaggle/working/species_site_counts.csv")

Saved:
/kaggle/working/train_ecological_metadata_features.csv
/kaggle/working/species_site_counts.csv


# Full Dataset Metadata Extraction

The previous section extracts metadata only for labeled train soundscape windows from `train_soundscapes_labels.csv`.

This section scans all `.ogg` files in the BirdCLEF 2026 input folder and extracts metadata from any filename that follows the soundscape pattern:

`BC2026_Train_0001_S08_20250606_030007.ogg`

or

`BC2026_Test_0001_S08_20250606_030007.ogg`

This helps separate:

1. labeled soundscape windows
2. all soundscape-style files
3. all audio files in the dataset

In [13]:
all_ogg_files = list(BASE.rglob("*.ogg"))

print("Total .ogg files found:", len(all_ogg_files))

audio_inventory = pd.DataFrame({
    "path": [str(p) for p in all_ogg_files],
    "relative_path": [str(p.relative_to(BASE)) for p in all_ogg_files],
    "filename": [p.name for p in all_ogg_files],
    "parent_folder": [p.parent.name for p in all_ogg_files],
})

display(audio_inventory.head())
print(audio_inventory["parent_folder"].value_counts().head(30))

Total .ogg files found: 46207


,path,relative_path,filename,parent_folder
0,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,train_soundscapes
1,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_9394_S22_202310...,BC2026_Train_9394_S22_20231027_204500.ogg,train_soundscapes
2,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_8559_S22_202212...,BC2026_Train_8559_S22_20221201_033000.ogg,train_soundscapes
3,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_7178_S22_202110...,BC2026_Train_7178_S22_20211014_034500.ogg,train_soundscapes
4,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_9248_S22_202310...,BC2026_Train_9248_S22_20231004_023000.ogg,train_soundscapes


parent_folder
train_soundscapes    10658
rubthr1                499
banana                 498
soulap1                497
fepowl                 497
houspa                 496
osprey                 495
coffal1                495
socfly1                494
compau                 493
yeofly1                493
bobfly1                492
bncfly                 492
whtdov                 491
trsowl                 491
bbwduc                 491
sobtyr1                490
roahaw                 489
strcuc1                485
trokin                 483
grekis                 482
saffin                 479
greyel                 475
greant1                475
brnowl                 474
barant1                454
sofspi1                437
gycwor1                436
squcuc1                432
oliwoo1                424
Name: count, dtype: int64


In [14]:
SOUNDSCAPE_FILE_RE = re.compile(
    r"BC2026_(Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg"
)

def parse_soundscape_file_metadata(filename):
    m = SOUNDSCAPE_FILE_RE.match(filename)

    if not m:
        return {
            "is_soundscape_style": False,
            "split": None,
            "file_id": None,
            "site": None,
            "date_str": None,
            "time_str": None,
            "datetime": pd.NaT,
            "year": np.nan,
            "month": np.nan,
            "day": np.nan,
            "hour": np.nan,
            "minute": np.nan,
            "second": np.nan,
            "day_of_year": np.nan,
        }

    split, file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd + hms, format="%Y%m%d%H%M%S")

    return {
        "is_soundscape_style": True,
        "split": split,
        "file_id": file_id,
        "site": site,
        "date_str": ymd,
        "time_str": hms,
        "datetime": dt,
        "year": dt.year,
        "month": dt.month,
        "day": dt.day,
        "hour": dt.hour,
        "minute": dt.minute,
        "second": dt.second,
        "day_of_year": dt.dayofyear,
    }

parsed_all_files = audio_inventory["filename"].apply(parse_soundscape_file_metadata).apply(pd.Series)
all_audio_metadata = pd.concat([audio_inventory, parsed_all_files], axis=1)

display(all_audio_metadata.head())
print("All audio files:", len(all_audio_metadata))
print("Soundscape-style files:", all_audio_metadata["is_soundscape_style"].sum())
print("Non-soundscape-style files:", (~all_audio_metadata["is_soundscape_style"]).sum())

,path,relative_path,filename,parent_folder,is_soundscape_style,split,file_id,site,date_str,time_str,datetime,year,month,day,hour,minute,second,day_of_year
0,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,train_soundscapes,True,Train,2754,S02,20211024,210000,2021-10-24 21:00:00,2021.0,10.0,24.0,21.0,0.0,0.0,297.0
1,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_9394_S22_202310...,BC2026_Train_9394_S22_20231027_204500.ogg,train_soundscapes,True,Train,9394,S22,20231027,204500,2023-10-27 20:45:00,2023.0,10.0,27.0,20.0,45.0,0.0,300.0
2,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_8559_S22_202212...,BC2026_Train_8559_S22_20221201_033000.ogg,train_soundscapes,True,Train,8559,S22,20221201,033000,2022-12-01 03:30:00,2022.0,12.0,1.0,3.0,30.0,0.0,335.0
3,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_7178_S22_202110...,BC2026_Train_7178_S22_20211014_034500.ogg,train_soundscapes,True,Train,7178,S22,20211014,034500,2021-10-14 03:45:00,2021.0,10.0,14.0,3.0,45.0,0.0,287.0
4,/kaggle/input/competitions/birdclef-2026/train...,train_soundscapes/BC2026_Train_9248_S22_202310...,BC2026_Train_9248_S22_20231004_023000.ogg,train_soundscapes,True,Train,9248,S22,20231004,023000,2023-10-04 02:30:00,2023.0,10.0,4.0,2.0,30.0,0.0,277.0


All audio files: 46207
Soundscape-style files: 10658
Non-soundscape-style files: 35549


In [15]:
all_audio_metadata.to_csv(
    "/kaggle/working/all_audio_file_inventory.csv",
    index=False
)

print("Saved /kaggle/working/all_audio_file_inventory.csv")

Saved /kaggle/working/all_audio_file_inventory.csv


In [16]:
soundscape_files = all_audio_metadata[
    all_audio_metadata["is_soundscape_style"]
].copy()

window_rows = []

for _, row in soundscape_files.iterrows():
    for start_second in range(0, 60, 5):
        end_second = start_second + 5

        row_id = (
            row["filename"].replace(".ogg", "")
            + "_"
            + str(end_second)
        )

        window_rows.append({
            "row_id": row_id,
            "relative_path": row["relative_path"],
            "filename": row["filename"],
            "split": row["split"],
            "file_id": row["file_id"],
            "site": row["site"],
            "datetime": row["datetime"],
            "year": row["year"],
            "month": row["month"],
            "day": row["day"],
            "hour": row["hour"],
            "minute": row["minute"],
            "second": row["second"],
            "day_of_year": row["day_of_year"],
            "start_second": start_second,
            "end_second": end_second,
            "window_index": start_second // 5,
        })

all_soundscape_windows = pd.DataFrame(window_rows)

display(all_soundscape_windows.head())
print("All soundscape-style files:", soundscape_files.shape[0])
print("All generated 5-second windows:", all_soundscape_windows.shape[0])

,row_id,relative_path,filename,split,file_id,site,datetime,year,month,day,hour,minute,second,day_of_year,start_second,end_second,window_index
0,BC2026_Train_2754_S02_20211024_210000_5,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,21.0,0.0,0.0,297.0,0,5,0
1,BC2026_Train_2754_S02_20211024_210000_10,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,21.0,0.0,0.0,297.0,5,10,1
2,BC2026_Train_2754_S02_20211024_210000_15,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,21.0,0.0,0.0,297.0,10,15,2
3,BC2026_Train_2754_S02_20211024_210000_20,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,21.0,0.0,0.0,297.0,15,20,3
4,BC2026_Train_2754_S02_20211024_210000_25,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,21.0,0.0,0.0,297.0,20,25,4


All soundscape-style files: 10658
All generated 5-second windows: 127896


In [17]:
all_soundscape_windows["hour_sin"] = np.sin(2 * np.pi * all_soundscape_windows["hour"] / 24)
all_soundscape_windows["hour_cos"] = np.cos(2 * np.pi * all_soundscape_windows["hour"] / 24)

all_soundscape_windows["month_sin"] = np.sin(2 * np.pi * all_soundscape_windows["month"] / 12)
all_soundscape_windows["month_cos"] = np.cos(2 * np.pi * all_soundscape_windows["month"] / 12)

all_soundscape_windows["dayofyear_sin"] = np.sin(2 * np.pi * all_soundscape_windows["day_of_year"] / 365)
all_soundscape_windows["dayofyear_cos"] = np.cos(2 * np.pi * all_soundscape_windows["day_of_year"] / 365)

display(all_soundscape_windows.head())

,row_id,relative_path,filename,split,file_id,site,datetime,year,month,day,...,day_of_year,start_second,end_second,window_index,hour_sin,hour_cos,month_sin,month_cos,dayofyear_sin,dayofyear_cos
0,BC2026_Train_2754_S02_20211024_210000_5,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,297.0,0,5,0,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963
1,BC2026_Train_2754_S02_20211024_210000_10,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,297.0,5,10,1,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963
2,BC2026_Train_2754_S02_20211024_210000_15,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,297.0,10,15,2,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963
3,BC2026_Train_2754_S02_20211024_210000_20,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,297.0,15,20,3,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963
4,BC2026_Train_2754_S02_20211024_210000_25,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,297.0,20,25,4,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963


In [18]:
labeled_row_ids = set(train_meta["row_id"])

all_soundscape_windows["has_label"] = all_soundscape_windows["row_id"].isin(labeled_row_ids)

print("All soundscape windows:", len(all_soundscape_windows))
print("Labeled soundscape windows:", all_soundscape_windows["has_label"].sum())
print("Unlabeled soundscape windows:", (~all_soundscape_windows["has_label"]).sum())

print("\nBy split:")
display(all_soundscape_windows.groupby(["split", "has_label"]).size().reset_index(name="count"))

print("\nBy site:")
display(all_soundscape_windows.groupby(["site", "has_label"]).size().reset_index(name="count"))

All soundscape windows: 127896
Labeled soundscape windows: 739
Unlabeled soundscape windows: 127157

By split:


,split,has_label,count
0,Train,False,127157
1,Train,True,739



By site:


,site,has_label,count
0,S01,False,28092
1,S02,False,30060
2,S03,False,36
3,S03,True,24
4,S04,False,204
5,S05,False,108
6,S06,False,648
7,S07,False,624
8,S08,True,60
9,S09,False,125


In [19]:
label_cols = ["row_id", "label_list"]

all_windows_with_labels = all_soundscape_windows.merge(
    train_meta[label_cols],
    on="row_id",
    how="left"
)

all_windows_with_labels["label_list"] = all_windows_with_labels["label_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

display(all_windows_with_labels.head())

,row_id,relative_path,filename,split,file_id,site,datetime,year,month,day,...,end_second,window_index,hour_sin,hour_cos,month_sin,month_cos,dayofyear_sin,dayofyear_cos,has_label,label_list
0,BC2026_Train_2754_S02_20211024_210000_5,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,5,0,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963,False,[]
1,BC2026_Train_2754_S02_20211024_210000_10,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,10,1,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963,False,[]
2,BC2026_Train_2754_S02_20211024_210000_15,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,15,2,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963,False,[]
3,BC2026_Train_2754_S02_20211024_210000_20,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,20,3,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963,False,[]
4,BC2026_Train_2754_S02_20211024_210000_25,train_soundscapes/BC2026_Train_2754_S02_202110...,BC2026_Train_2754_S02_20211024_210000.ogg,Train,2754,S02,2021-10-24 21:00:00,2021.0,10.0,24.0,...,25,4,-0.707107,0.707107,-0.866025,0.5,-0.920971,0.38963,False,[]


In [20]:
all_soundscape_windows.to_csv(
    "/kaggle/working/all_soundscape_window_metadata.csv",
    index=False
)

all_windows_with_labels.to_csv(
    "/kaggle/working/all_soundscape_window_metadata_with_labels.csv",
    index=False
)

print("Saved:")
print("/kaggle/working/all_audio_file_inventory.csv")
print("/kaggle/working/all_soundscape_window_metadata.csv")
print("/kaggle/working/all_soundscape_window_metadata_with_labels.csv")

Saved:
/kaggle/working/all_audio_file_inventory.csv
/kaggle/working/all_soundscape_window_metadata.csv
/kaggle/working/all_soundscape_window_metadata_with_labels.csv


In [21]:
# Create a simplified ecological metadata format matching Prof. Yin's example

def make_simple_metadata_format(df, include_labels=True):
    simple = pd.DataFrame()

    simple["row_id"] = df["row_id"]
    simple["site"] = df["site"]

    extracted = df["row_id"].str.extract(
        r"BC2026_(?:Train|Test)_\d+_S\d+_(\d{8})_(\d{6})_\d+"
    )

    simple["date/time"] = extracted[0] + "_" + extracted[1]
    simple["end second"] = df["end_second"].astype(int)
    simple["start second"] = df["start_second"].astype(int)

    if "filename" in df.columns:
        simple["filename"] = df["filename"]

    if "split" in df.columns:
        simple["split"] = df["split"]

    if "has_label" in df.columns:
        simple["has_label"] = df["has_label"]

    if include_labels and "label_list" in df.columns:
        simple["label_list"] = df["label_list"]

    return simple


train_simple = make_simple_metadata_format(train_meta, include_labels=True)

train_simple.to_csv(
    "/kaggle/working/train_ecological_metadata_simple_format.csv",
    index=False
)

print("Saved /kaggle/working/train_ecological_metadata_simple_format.csv")
display(train_simple.head())

Saved /kaggle/working/train_ecological_metadata_simple_format.csv


,row_id,site,date/time,end second,start second,filename,split,label_list
0,BC2026_Train_0001_S08_20250606_030007_5,S08,20250606_030007,5,0,BC2026_Train_0001_S08_20250606_030007.ogg,Train,"[47158son13, 47158son17, 47158son22, 47158son2..."
1,BC2026_Train_0001_S08_20250606_030007_10,S08,20250606_030007,10,5,BC2026_Train_0001_S08_20250606_030007.ogg,Train,"[47158son13, 47158son17, 47158son21, 47158son2..."
2,BC2026_Train_0001_S08_20250606_030007_15,S08,20250606_030007,15,10,BC2026_Train_0001_S08_20250606_030007.ogg,Train,"[47158son13, 47158son17, 47158son21, 47158son2..."
3,BC2026_Train_0001_S08_20250606_030007_20,S08,20250606_030007,20,15,BC2026_Train_0001_S08_20250606_030007.ogg,Train,"[47158son13, 47158son17, 47158son21, 47158son2..."
4,BC2026_Train_0001_S08_20250606_030007_25,S08,20250606_030007,25,20,BC2026_Train_0001_S08_20250606_030007.ogg,Train,"[47158son10, 47158son13, 47158son17, 47158son2..."


In [22]:
all_simple = make_simple_metadata_format(
    all_windows_with_labels,
    include_labels=True
)

all_simple.to_csv(
    "/kaggle/working/all_soundscape_window_metadata_simple_format.csv",
    index=False
)

print("Saved /kaggle/working/all_soundscape_window_metadata_simple_format.csv")
display(all_simple.head())

Saved /kaggle/working/all_soundscape_window_metadata_simple_format.csv


,row_id,site,date/time,end second,start second,filename,split,has_label,label_list
0,BC2026_Train_2754_S02_20211024_210000_5,S02,20211024_210000,5,0,BC2026_Train_2754_S02_20211024_210000.ogg,Train,False,[]
1,BC2026_Train_2754_S02_20211024_210000_10,S02,20211024_210000,10,5,BC2026_Train_2754_S02_20211024_210000.ogg,Train,False,[]
2,BC2026_Train_2754_S02_20211024_210000_15,S02,20211024_210000,15,10,BC2026_Train_2754_S02_20211024_210000.ogg,Train,False,[]
3,BC2026_Train_2754_S02_20211024_210000_20,S02,20211024_210000,20,15,BC2026_Train_2754_S02_20211024_210000.ogg,Train,False,[]
4,BC2026_Train_2754_S02_20211024_210000_25,S02,20211024_210000,25,20,BC2026_Train_2754_S02_20211024_210000.ogg,Train,False,[]
